# Gemma 4 E4B agent, fuller settings, on a Colab GPU (spec 035 US2, #656)

**Runtime -> Change runtime type -> A100 GPU** (L4 works, with a shorter context). Then **Runtime -> Run all**.
With Colab Pro+ you can close the tab: it keeps running (up to 24 hours). On Pro, keep the tab open.

**Once, before Run all:** the key icon on the left (Secrets), add two secrets and turn on *Notebook access* for both:
`KAGGLE_USERNAME` = `jcblearning`, and `KAGGLE_API_TOKEN` = your Kaggle API token (kaggle.com -> Settings -> API ->
Generate New Token). Never paste the token into a cell.

Each task runs alone: its private bundle (no fix, no tests) is downloaded, the agent runs, the result is uploaded to
the private dataset `jcblearning/htlnlp-colab-results`, and the bundle is deleted before the next one. A task already
in the results is skipped, so Run all again after a disconnect carries on. The patches are scored back in the
repository (`python/scripts/62_colab_agent.py score`). Measured with the repository's harness, not the competition's
evaluator.

In [ ]:
import os, subprocess
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
assert os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_API_TOKEN"], "add both Colab secrets (see above)"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate kagglehub "kaggle==2.2.5" six

In [ ]:
import glob, json, shutil, sys, time, traceback, zipfile
from pathlib import Path
import torch, kagglehub
from kaggle.api.kaggle_api_extended import KaggleApi

IDS = "htlnlp_230,htlnlp_120,htlnlp_pr63,htlnlp_152,htlnlp_189,htlnlp_pr35,htlnlp_226,htlnlp_pr21,htlnlp_c7300f3f,htlnlp_pr36,htlnlp_pr53,htlnlp_pr31,htlnlp_pr32,htlnlp_111,htlnlp_pr30,htlnlp_pr57,htlnlp_pr23,htlnlp_pr55,htlnlp_187,htlnlp_94,htlnlp_pr66,htlnlp_pr37,htlnlp_155,htlnlp_pr56,htlnlp_cecf7a16,htlnlp_244,htlnlp_228,htlnlp_pr34,htlnlp_c59170ee,htlnlp_90,htlnlp_144,htlnlp_116,htlnlp_143,htlnlp_170,htlnlp_174,htlnlp_192,htlnlp_pr47,htlnlp_pr17,htlnlp_c791d60f,htlnlp_71,htlnlp_pr33,htlnlp_cfa7c008,htlnlp_249,htlnlp_pr29,htlnlp_255,htlnlp_236,htlnlp_206,htlnlp_276".split(",")
RESULTS = "jcblearning/htlnlp-colab-results"
api = KaggleApi(); api.authenticate()
api.dataset_list(user="jcblearning", page=1)          # one cheap call: the token works
mem_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
MAX_CONTEXT = 32768 if mem_gib >= 39 else 20000      # an L4's 24 GB holds less beside the weights
print(f"{torch.cuda.get_device_name(0)}, {mem_gib:.0f} GiB -> max_context {MAX_CONTEXT}")
model_dir = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e4b-it/1")
print("model at", model_dir)

In [ ]:
res = Path("/content/results"); res.mkdir(exist_ok=True)
try:
    api.dataset_download_files(RESULTS, path=str(res), unzip=True, quiet=True)
except Exception as e:
    print("no results yet:", type(e).__name__)
done = {p.stem for p in res.glob("*.json")}
todo = [i for i in IDS if i not in done]
print(f"{len(done)} done, {len(todo)} to run")
(res / "dataset-metadata.json").write_text(json.dumps({"id": RESULTS, "title": "htlnlp colab results",
                                                       "licenses": [{"name": "CC-BY-4.0"}]}))

def slug(i):
    import re
    return "htlnlp-colab-" + re.sub(r"[^a-z0-9]+", "-", i.lower()).strip("-")

model = None
for n, iid in enumerate(todo, 1):
    t0 = time.monotonic()
    bundle = Path("/content/task")
    shutil.rmtree(bundle, ignore_errors=True); shutil.rmtree("/tmp/htl_work", ignore_errors=True)
    result = {"instance_id": iid, "started_ts": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    try:
        for wait in range(30):   # the bundles upload while this runs: wait for one not there yet (up to 30 min)
            try:
                api.dataset_download_files(f"jcblearning/{slug(iid)}", path=str(bundle), unzip=True, quiet=True)
                break
            except Exception:
                if wait == 29:
                    raise
                print(f"{iid}: bundle not ready, waiting"); time.sleep(60)
        if model is None:   # the harness the bundles ship; loaded once, the model too
            sys.path.insert(0, str(bundle))
            from htl_agent import agent_harness as ah, gemma_agent as ga
            import numpy as np
        task = json.loads((bundle / "task.json").read_text())
        cfg = json.loads((bundle / "agent.json").read_text())
        mcfg = dict(cfg["model"]); variant = mcfg.pop("variant")
        mcfg["max_context"] = MAX_CONTEXT
        if model is None:
            t = time.monotonic()
            model = ga.GemmaModel(model_dir, name=variant, **mcfg)
            result["load_seconds"] = round(time.monotonic() - t, 1)
        model.cache, model.ids, model.seen, model.turns, model.history = None, [], 0, [], []
        graph = None
        if (bundle / "graph.json").exists():
            vec = dict(np.load(bundle / "embedding.npz")) if (bundle / "embedding.npz").exists() else {}
            graph = ah.CodeGraph(json.loads((bundle / "graph.json").read_text()), vec)
        row, patch = ah.agent_phase(task, bundle / "snapshot.tgz", model, ah.Budget(**cfg["budget"]),
                                    Path("/tmp/htl_work"), cfg["instruction"], cfg["tools"], graph, sandboxed=False)
        result.update(row=row, patch=patch, turns=model.turns, history=model.history,
                      gpus=[torch.cuda.get_device_name(0)],
                      settings={"variant": variant, **mcfg, "budget": cfg["budget"], "tools": cfg["tools"]})
    except Exception as e:
        result.update(error=f"{type(e).__name__}: {str(e)[:1000]}", trace=traceback.format_exc()[-3000:])
        print(result["trace"])
    finally:
        torch.cuda.empty_cache()
    result["finished_ts"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    result["task_seconds"] = round(time.monotonic() - t0, 1)
    (res / f"{iid}.json").write_text(json.dumps(result, default=str))
    shutil.rmtree(bundle, ignore_errors=True)
    for attempt in range(4):   # Kaggle sometimes drops an upload; the result is on disk either way
        try:
            api.dataset_create_version(str(res), f"{iid}", dir_mode="zip", quiet=True)
            break
        except Exception as e:
            print("upload failed, retrying:", type(e).__name__); time.sleep(60)
    r = result.get("row") or {}
    print(f"[{n}/{len(todo)}] {iid}: patch {len(result.get('patch') or '')} bytes, {r.get('tool_call_count')} calls, "
          f"{result['task_seconds'] / 60:.1f} min{' ERROR' if 'error' in result else ''}", flush=True)
print("all done")